# 04 Encoding and Train/Validation Split (Milestone 2, Task 2)

Input: `featured_df.csv` from Task 1.

Task: turn the categorical columns into numbers the models can use, then split the data into training and validation sets.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Works from the repo root or from the notebooks/ folder
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
IN_FILE = ROOT / "data" / "processed" / "featured_df.csv"
OUT_DIR = ROOT / "data" / "processed" / "train_val_split"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "cpe"
DATE_COL = "campaign_start_date"
VAL_SHARE = 0.20   # newest 20% of campaigns go to validation
MIN_FREQ = 5       # categories seen fewer than 5 times in training share one "infrequent" column

df = pd.read_csv(IN_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
print(df.shape)

(417, 60)


## 1. Identify the categorical and numeric columns

In [2]:
CAT_COLS = [c for c in df.columns
            if not pd.api.types.is_numeric_dtype(df[c]) and c != DATE_COL]
NUM_COLS = [c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c]) and c != TARGET]

print(len(CAT_COLS), "categorical columns:")
print(df[CAT_COLS].nunique().sort_values(ascending=False))
print()
print(len(NUM_COLS), "numeric columns (kept as they are)")

14 categorical columns:
brand_raw           73
company             34
product             23
profession          14
npi_specialty       14
sub_product_name    11
region               9
country              9
specialty            9
country_name         9
channel_tag          8
pillar               5
product_tier         4
media_type           3
dtype: int64

44 numeric columns (kept as they are)


## 2. Split into training and validation sets

The split is time-based. Campaigns are sorted by start date, the oldest 80% go to training and the newest 20% go to validation. The cut is made on a date, so campaigns that start on the same day stay in the same set.

In [3]:
cutoff = df[DATE_COL].quantile(1 - VAL_SHARE)
is_val = df[DATE_COL] > cutoff
train, val = df[~is_val], df[is_val]

print("cutoff date:", cutoff.date())
print(f"train: {len(train)} rows  {train[DATE_COL].min().date()} to {train[DATE_COL].max().date()}")
print(f"val:   {len(val)} rows  {val[DATE_COL].min().date()} to {val[DATE_COL].max().date()}")
assert train[DATE_COL].max() < val[DATE_COL].min()

cutoff date: 2025-03-11
train: 335 rows  2020-12-17 to 2025-03-11
val:   82 rows  2025-03-12 to 2026-04-17


## 3. One-hot encode the categorical columns

* The encoder is fit on the training rows only, then applied to both sets.
* Categories seen fewer than 5 times in training are grouped into one `..._infrequent_sklearn` column per categorical column.
* A category that appears only in validation goes into that infrequent column. If a column has no infrequent group, the row gets all zeros for that column.

In [4]:
enc = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=MIN_FREQ,
                    sparse_output=False, dtype=int)
enc.fit(train[CAT_COLS])

def encode(part):
    dummies = pd.DataFrame(enc.transform(part[CAT_COLS]),
                           columns=enc.get_feature_names_out(CAT_COLS), index=part.index)
    return pd.concat([part[NUM_COLS], dummies], axis=1)

X_train, X_val = encode(train), encode(val)
y_train, y_val = train[TARGET], val[TARGET]

print(f"{len(enc.get_feature_names_out(CAT_COLS))} dummy columns from {len(CAT_COLS)} categorical columns")
print("X_train:", X_train.shape, " X_val:", X_val.shape)

117 dummy columns from 14 categorical columns
X_train: (335, 161)  X_val: (82, 161)


## 4. Checks

In [5]:
assert X_train.isna().sum().sum() == 0 and X_val.isna().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train.dtypes)
assert list(X_train.columns) == list(X_val.columns)
assert TARGET not in X_train.columns
assert len(X_train) + len(X_val) == len(df)
print("all checks passed")

all checks passed


## 5. Save

In [6]:
X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
X_val.to_csv(OUT_DIR / "X_val.csv", index=False)
y_train.to_frame().to_csv(OUT_DIR / "y_train.csv", index=False)
y_val.to_frame().to_csv(OUT_DIR / "y_val.csv", index=False)

for p in sorted(OUT_DIR.glob("*.csv")):
    print(p.relative_to(ROOT))

data/processed/train_val_split/X_train.csv
data/processed/train_val_split/X_val.csv
data/processed/train_val_split/y_train.csv
data/processed/train_val_split/y_val.csv
